In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import matplotlib.pyplot as plt
import seaborn as sns


class RealWorldMICE:
    """
    Professional MICE implementation for real-world datasets
    Handles mixed data types, large features, and various data quality issues
    """

    def __init__(self,
                 max_iter=20,
                 tol=1e-3,
                 missing_threshold=0.95,
                 feature_selection=True,
                 max_features=50,
                 handle_outliers=True,
                 verbose=True,
                 random_state=42):
        """
        Parameters:
        -----------
        max_iter : int, maximum iterations for MICE
        tol : float, tolerance for convergence
        missing_threshold : float, drop columns with missing > this threshold
        feature_selection : bool, whether to select important features
        max_features : int, maximum features to use for imputation
        handle_outliers : bool, whether to handle outliers
        verbose : bool, whether to print progress
        random_state : int, random seed
        """
        self.max_iter = max_iter
        self.tol = tol
        self.missing_threshold = missing_threshold
        self.feature_selection = feature_selection
        self.max_features = max_features
        self.handle_outliers = handle_outliers
        self.verbose = verbose
        self.random_state = random_state

        # Storage for fitted objects
        self.label_encoders_ = {}
        self.scalers_ = {}
        self.imputers_ = {}
        self.feature_selector_ = None
        self.selected_features_ = None
        self.column_types_ = {}

    def _print(self, message):
        """Print if verbose is True"""
        if self.verbose:
            print(message)

    def _detect_column_types(self, df):
        """Detect and categorize column types"""
        self._print("\n🔍 DETECTING COLUMN TYPES")
        self._print("=" * 40)

        numerical_cols = []
        categorical_cols = []
        datetime_cols = []
        high_cardinality_cols = []

        for col in df.columns:
            # Check for datetime
            if df[col].dtype == 'datetime64[ns]' or 'date' in col.lower():
                datetime_cols.append(col)

            # Check for numerical
            elif df[col].dtype in ['int64', 'float64', 'int32', 'float32']:
                numerical_cols.append(col)

            # Check for categorical
            else:
                unique_ratio = df[col].nunique() / len(df)

                # High cardinality (like IDs) - usually not useful for imputation
                if unique_ratio > 0.8:
                    high_cardinality_cols.append(col)
                else:
                    categorical_cols.append(col)

        self.column_types_ = {
            'numerical': numerical_cols,
            'categorical': categorical_cols,
            'datetime': datetime_cols,
            'high_cardinality': high_cardinality_cols
        }

        self._print(
            f"📊 Numerical columns ({len(numerical_cols)}): {numerical_cols[:5]}{'...' if len(numerical_cols) > 5 else ''}")
        self._print(
            f"📊 Categorical columns ({len(categorical_cols)}): {categorical_cols[:5]}{'...' if len(categorical_cols) > 5 else ''}")
        self._print(
            f"📊 Datetime columns ({len(datetime_cols)}): {datetime_cols}")
        self._print(
            f"📊 High cardinality columns ({len(high_cardinality_cols)}): {high_cardinality_cols[:3]}{'...' if len(high_cardinality_cols) > 3 else ''}")

        return self.column_types_

    def _handle_missing_analysis(self, df):
        """Analyze missing data patterns"""
        self._print("\n📊 MISSING DATA ANALYSIS")
        self._print("=" * 40)

        missing_stats = pd.DataFrame({
            'Column': df.columns,
            'Missing_Count': df.isnull().sum(),
            'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
            'Data_Type': df.dtypes
        }).sort_values('Missing_Percentage', ascending=False)

        # Identify columns to drop
        cols_to_drop = missing_stats[missing_stats['Missing_Percentage']
                                     > self.missing_threshold * 100]['Column'].tolist()

        self._print(f"📋 Missing data summary (top 10):")
        print(missing_stats.head(10).to_string(index=False))

        if cols_to_drop:
            self._print(
                f"\n❌ Dropping {len(cols_to_drop)} columns with >{self.missing_threshold*100}% missing data:")
            self._print(f"   {cols_to_drop}")

        return cols_to_drop, missing_stats

    def _handle_outliers(self, df, numerical_cols):
        """Handle outliers using IQR method"""
        if not self.handle_outliers or not numerical_cols:
            return df

        self._print("\n🔧 HANDLING OUTLIERS")
        self._print("=" * 30)

        df_clean = df.copy()
        outlier_stats = {}

        for col in numerical_cols:
            if col not in df_clean.columns:
                continue

            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1

            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            outlier_mask = (df_clean[col] < lower_bound) | (
                df_clean[col] > upper_bound)
            outlier_count = outlier_mask.sum()

            if outlier_count > 0:
                # Cap outliers instead of removing them
                df_clean.loc[df_clean[col] < lower_bound, col] = lower_bound
                df_clean.loc[df_clean[col] > upper_bound, col] = upper_bound

                outlier_stats[col] = outlier_count

        if outlier_stats:
            self._print(f"🔧 Capped outliers in {len(outlier_stats)} columns:")
            for col, count in outlier_stats.items():
                self._print(f"   {col}: {count} outliers capped")

        return df_clean

    def _encode_categorical_variables(self, df, categorical_cols):
        """Encode categorical variables with proper handling"""
        if not categorical_cols:
            return df

        self._print("\n🔄 ENCODING CATEGORICAL VARIABLES")
        self._print("=" * 40)

        df_encoded = df.copy()

        for col in categorical_cols:
            if col not in df_encoded.columns:
                continue

            # Handle missing values first
            non_null_mask = df_encoded[col].notna()
            if non_null_mask.sum() == 0:
                self._print(f"⚠️ Skipping {col}: all values are missing")
                continue

            # Check cardinality
            unique_values = df_encoded[col].dropna().nunique()
            if unique_values > 50:
                self._print(
                    f"⚠️ High cardinality in {col}: {unique_values} unique values")
                # For high cardinality, use frequency encoding
                freq_map = df_encoded[col].value_counts().to_dict()
                df_encoded[col] = df_encoded[col].map(freq_map)
            else:
                # Standard label encoding
                le = LabelEncoder()
                df_encoded.loc[non_null_mask, col] = le.fit_transform(
                    df_encoded.loc[non_null_mask, col])
                self.label_encoders_[col] = le

            self._print(f"✅ Encoded {col}: {unique_values} categories")

        return df_encoded

    def _feature_selection(self, df, target_col=None):
        """Select most important features for imputation"""
        if not self.feature_selection or len(df.columns) <= self.max_features:
            return df

        self._print(
            f"\n🎯 FEATURE SELECTION (reducing from {len(df.columns)} to {self.max_features})")
        self._print("=" * 50)

        # If no target provided, use correlation-based selection
        if target_col is None or target_col not in df.columns:
            # Calculate correlation matrix and select features with highest average correlation
            corr_matrix = df.corr().abs()
            avg_corr = corr_matrix.mean().sort_values(ascending=False)
            selected_features = avg_corr.head(self.max_features).index.tolist()

            self._print(f"📊 Selected features based on average correlation:")
            for i, feat in enumerate(selected_features[:10]):
                self._print(f"   {i+1:2d}. {feat}: {avg_corr[feat]:.3f}")

        else:
            # Use target-based feature selection
            try:
                selector = SelectKBest(score_func=f_regression, k=min(
                    self.max_features, len(df.columns)-1))
                X = df.drop(columns=[target_col])
                y = df[target_col]

                # Handle missing values for feature selection
                X_temp = X.fillna(X.mean(numeric_only=True))
                selector.fit(X_temp, y)

                selected_features = X.columns[selector.get_support()].tolist()
                if target_col not in selected_features:
                    selected_features.append(target_col)

                self._print(f"📊 Selected features based on target correlation")

            except Exception as e:
                self._print(f"⚠️ Target-based selection failed: {e}")
                # Fallback to correlation-based
                corr_matrix = df.corr().abs()
                avg_corr = corr_matrix.mean().sort_values(ascending=False)
                selected_features = avg_corr.head(
                    self.max_features).index.tolist()

        self.selected_features_ = selected_features
        return df[selected_features]

    def fit_transform(self, df, target_col=None):
        """
        Main method to fit and transform the dataset
        
        Parameters:
        -----------
        df : pandas.DataFrame, input dataset
        target_col : str, target column name (if any)
        
        Returns:
        --------
        pandas.DataFrame, imputed dataset
        """
        self._print("🚀 STARTING REAL-WORLD MICE IMPUTATION")
        self._print("=" * 60)
        self._print(f"📊 Input dataset shape: {df.shape}")
        self._print(f"📊 Missing values: {df.isnull().sum().sum()}")

        # Step 1: Detect column types
        self._detect_column_types(df)

        # Step 2: Analyze missing data and drop problematic columns
        cols_to_drop, missing_stats = self._handle_missing_analysis(df)

        # Drop high cardinality and mostly missing columns
        all_drops = cols_to_drop + self.column_types_['high_cardinality']
        df_clean = df.drop(
            columns=[col for col in all_drops if col in df.columns])

        self._print(
            f"\n📊 After dropping problematic columns: {df_clean.shape}")

        # Step 3: Handle datetime columns (extract features)
        for col in self.column_types_['datetime']:
            if col in df_clean.columns:
                df_clean[f'{col}_year'] = pd.to_datetime(df_clean[col]).dt.year
                df_clean[f'{col}_month'] = pd.to_datetime(
                    df_clean[col]).dt.month
                df_clean[f'{col}_day'] = pd.to_datetime(df_clean[col]).dt.day
                df_clean = df_clean.drop(columns=[col])
                self._print(f"📅 Extracted features from {col}")

        # Update column types after datetime processing
        self._detect_column_types(df_clean)

        # Step 4: Handle outliers
        df_clean = self._handle_outliers(
            df_clean, self.column_types_['numerical'])

        # Step 5: Encode categorical variables
        df_encoded = self._encode_categorical_variables(
            df_clean, self.column_types_['categorical'])

        # Step 6: Feature selection
        df_selected = self._feature_selection(df_encoded, target_col)

        # Step 7: Scale numerical features for better convergence
        numerical_cols_final = [col for col in self.column_types_[
            'numerical'] if col in df_selected.columns]
        if numerical_cols_final:
            self._print(f"\n📏 SCALING NUMERICAL FEATURES")
            scaler = RobustScaler()  # More robust to outliers than StandardScaler
            df_selected[numerical_cols_final] = scaler.fit_transform(
                df_selected[numerical_cols_final])
            self.scalers_['numerical'] = scaler
            self._print(
                f"✅ Scaled {len(numerical_cols_final)} numerical columns")

        # Step 8: Apply MICE imputation
        self._print(f"\n🔄 APPLYING MICE IMPUTATION")
        self._print("=" * 40)

        # Choose appropriate estimator based on data characteristics
        if len(df_selected.columns) > 20:
            estimator = RandomForestRegressor(
                n_estimators=10, max_depth=5, random_state=self.random_state)
            self._print("🌲 Using RandomForest (many features)")
        else:
            estimator = LinearRegression()
            self._print("📈 Using LinearRegression (few features)")

        # Apply MICE
        mice_imputer = IterativeImputer(
            estimator=estimator,
            max_iter=self.max_iter,
            tol=self.tol,
            imputation_order='ascending',
            verbose=1 if self.verbose else 0,
            random_state=self.random_state
        )

        try:
            df_imputed = mice_imputer.fit_transform(df_selected)
            df_imputed = pd.DataFrame(
                df_imputed, columns=df_selected.columns, index=df_selected.index)

            self.imputers_['mice'] = mice_imputer

            # Check convergence
            iterations_used = len(
                mice_imputer.imputation_sequence_) // len(df_selected.columns)
            converged = iterations_used < self.max_iter

            self._print(f"\n✅ MICE COMPLETED!")
            self._print(
                f"📊 Iterations used: {iterations_used}/{self.max_iter}")
            self._print(f"📊 Converged: {'Yes' if converged else 'No'}")

        except Exception as e:
            self._print(f"\n❌ MICE failed: {e}")
            self._print("🔄 Falling back to SimpleImputer")

            # Fallback to simple imputation
            num_imputer = SimpleImputer(strategy='median')
            cat_imputer = SimpleImputer(strategy='most_frequent')

            df_imputed = df_selected.copy()

            if numerical_cols_final:
                df_imputed[numerical_cols_final] = num_imputer.fit_transform(
                    df_imputed[numerical_cols_final])

            categorical_cols_final = [col for col in self.column_types_[
                'categorical'] if col in df_selected.columns]
            if categorical_cols_final:
                df_imputed[categorical_cols_final] = cat_imputer.fit_transform(
                    df_imputed[categorical_cols_final])

        # Step 9: Inverse transform scaling
        if numerical_cols_final and 'numerical' in self.scalers_:
            df_imputed[numerical_cols_final] = self.scalers_[
                'numerical'].inverse_transform(df_imputed[numerical_cols_final])
            self._print("✅ Inverse scaled numerical features")

        # Step 10: Decode categorical variables
        for col, encoder in self.label_encoders_.items():
            if col in df_imputed.columns:
                # Round to nearest integer and clip to valid range
                df_imputed[col] = df_imputed[col].round().astype(int)
                df_imputed[col] = df_imputed[col].clip(
                    0, len(encoder.classes_) - 1)
                df_imputed[col] = encoder.inverse_transform(df_imputed[col])
                self._print(f"✅ Decoded {col}")

        self._print(f"\n🎉 FINAL RESULT: {df_imputed.shape}")
        self._print(
            f"📊 Missing values remaining: {df_imputed.isnull().sum().sum()}")

        return df_imputed

    def get_feature_importance(self):
        """Get feature importance from the imputation process"""
        if 'mice' not in self.imputers_:
            return None

        # This is a simplified importance - in practice, you'd analyze
        # the coefficients or feature importance from each iteration
        return f"Used {len(self.selected_features_) if self.selected_features_ else 'all'} features for imputation"

    def plot_imputation_diagnostics(self, original_df, imputed_df):
        """Plot diagnostic plots for imputation quality"""
        numerical_cols = [col for col in self.column_types_[
            'numerical'] if col in imputed_df.columns]

        if not numerical_cols:
            self._print("No numerical columns to plot")
            return

        n_cols = min(3, len(numerical_cols))
        fig, axes = plt.subplots(2, n_cols, figsize=(5*n_cols, 8))

        if n_cols == 1:
            axes = axes.reshape(-1, 1)

        for i, col in enumerate(numerical_cols[:n_cols]):
            # Distribution comparison
            axes[0, i].hist(original_df[col].dropna(), alpha=0.7,
                            label='Original', bins=20, density=True)
            axes[0, i].hist(imputed_df[col], alpha=0.7,
                            label='Imputed', bins=20, density=True)
            axes[0, i].set_title(f'{col} - Distribution')
            axes[0, i].legend()
            axes[0, i].grid(True, alpha=0.3)

            # Missing pattern
            missing_mask = original_df[col].isnull()
            if missing_mask.sum() > 0:
                axes[1, i].scatter(
                    range(len(original_df)), original_df[col], alpha=0.6, label='Original', s=10)
                axes[1, i].scatter(np.where(missing_mask)[0], imputed_df[col][missing_mask],
                                   alpha=0.8, label='Imputed', color='red', s=15)
                axes[1, i].set_title(f'{col} - Imputed Values')
                axes[1, i].legend()
                axes[1, i].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# Example usage with simulated real-world data


def create_realistic_dataset():
    """Create a realistic dataset with common real-world problems"""
    np.random.seed(42)
    n_samples = 1000

    # Create dataset with various data types and problems
    data = {
        # Numerical features
        'age': np.random.normal(35, 12, n_samples),
        'income': np.random.lognormal(10, 1, n_samples),
        'score': np.random.beta(2, 5, n_samples) * 100,
        'experience': np.random.poisson(8, n_samples),
        'rating': np.random.uniform(1, 5, n_samples),

        # Categorical features
        'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
        'department': np.random.choice(['IT', 'HR', 'Finance', 'Marketing', 'Sales'], n_samples),
        # High cardinality
        'city': np.random.choice([f'City_{i}' for i in range(50)], n_samples),

        # Target
        'target': np.random.binomial(1, 0.3, n_samples)
    }

    df = pd.DataFrame(data)

    # Introduce realistic missing patterns
    missing_patterns = {
        'age': 0.1,           # 10% missing
        'income': 0.15,       # 15% missing
        'score': 0.05,        # 5% missing
        'education': 0.08,    # 8% missing
        'department': 0.02,   # 2% missing
        'city': 0.12,         # 12% missing
        'rating': 0.20        # 20% missing
    }

    for col, missing_rate in missing_patterns.items():
        missing_idx = np.random.choice(n_samples, int(
            missing_rate * n_samples), replace=False)
        df.loc[missing_idx, col] = np.nan

    # Add some extreme outliers
    outlier_idx = np.random.choice(n_samples, 50, replace=False)
    df.loc[outlier_idx, 'income'] = df.loc[outlier_idx,
                                           'income'] * 10  # Income outliers

    return df


# Test the function
print("🧪 TESTING WITH REALISTIC DATASET")
print("=" * 50)

# Create realistic dataset
test_df = create_realistic_dataset()
print(f"📊 Created realistic dataset: {test_df.shape}")
print(f"📊 Missing values: {test_df.isnull().sum().sum()}")

# Apply real-world MICE
mice_processor = RealWorldMICE(
    max_iter=15,
    tol=1e-3,
    missing_threshold=0.8,
    feature_selection=True,
    max_features=10,
    handle_outliers=True,
    verbose=True
)

# Process the data
result_df = mice_processor.fit_transform(test_df, target_col='target')

# Show results
print(f"\n📈 FINAL COMPARISON:")
print(f"Original shape: {test_df.shape}")
print(f"Final shape: {result_df.shape}")
print(f"Original missing: {test_df.isnull().sum().sum()}")
print(f"Final missing: {result_df.isnull().sum().sum()}")

# Plot diagnostics
mice_processor.plot_imputation_diagnostics(test_df, result_df)